<a href="https://colab.research.google.com/github/wangwangwang77/Machine-learning-in-UM/blob/main/main_code_pipeline_v5_fast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline v5-fast:回归 v5 方法 + 纯工程提速(算法不变)

**方法与 v5 逐字相同**(纯端到端 NN:【修正 1–3】+【改动 A/B/D/E/G/I】,三阶段块坐标训练;BI 仅为外部基准)。本版只做三处不改变算法的工程优化(标记【S1–S3】),目标是把 T4 上 ~2 小时的运行压到 ~20–30 分钟: $\gamma$=5

| 标记 | 优化 | 说明 |
|---|---|---|
| S1 | 验证评估包进 `tf.function` | 此前每 epoch 在全部验证路径上以 eager 方式重建计算图,是运行时间的主要去向;现为单次图调用 |
| S2 | 训练批量 ×`BATCH_MUL`(默认 4) | GPU 在 200 路径/步时严重欠载;更大批量下**梯度期望不变**、噪声更小、步数等比减少,近线性提速 |
| S3 | 训练中无任何 `collect=True` 逐期监控 | 逐期路径收集只在最终验证节执行一次 |

**penalty 系列实验的归档结论**(可直接写入论文 ablation):pen4 验证了 latch 原理在容量允许的方向上成立(退休期出现被锁存的水平台阶,经 $Y/Y_0=0$ 指示器通道),但退休段内部的时间剖面(ramp)无法经标量 ttm 由共享网络表达——生成剖面所需的正交容量恰为逐期参数化(v5 的 $\varphi$)所提供;且准静态 $\lambda$–$\eta$ 联动日程对工作期与组合收敛造成附带损害。故论文方法采用 v5。

## 0. 环境

In [ ]:
!pip install -q openpyxl
import numpy as np, tensorflow as tf
print('TF', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## 1. 数据(逐字相同)

In [ ]:
# =============================================================================
# 第 1 步:数据 —— 从 Google Drive 读取原始 Excel,构造联合样本,做 bootstrap
# 数据地址与原代码完全一致
# =============================================================================
import numpy as np
import pandas as pd
from functools import reduce
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# ---------------------------- 全局参数 ---------------------------------------
GAMMA        = 5.0          # 相对风险厌恶系数
BETA         = 0.96         # 主观贴现因子
T            = 60           # 生命周期总期数
K_EMPLOY     = 40           # 工作期数(其后退休、无劳动收入)
START_WEALTH = 0.0
START_INCOME = 10.0
X0           = START_WEALTH + START_INCOME     # 期初 cash-on-hand

INDUSTRY_ID  = 1            # 0..7, 1 = Finance
N_PATH       = 200          # 每个 batch 的路径数
N_BATCH_TR   = 250
N_BATCH_VA   = 100
N_BATCH_TE   = 100
N_BATCH      = N_BATCH_TR + N_BATCH_VA + N_BATCH_TE
SEED         = 7

# 各行业在不同表中的列名映射(与原始 Excel 的列名一一对应)
IND_RET  = ['Construction_Ip','Finance_Ip','Manuf_Ip','Mining_Ip',
            'retail_Ip','Service_Ip','Comm. Util._Ip','wholesale_Ip']
IND_MK   = ['R_M_excl_Construction','R_M_excl_Finance','R_M_excl_Manuf',
            'R_M_excl_Mining','R_M_excl_retail','R_M_excl_Service',
            'R_M_excl_Comm. Util.','R_M_excl_wholesale']
IND_DP   = ['dp_constr','Fin_dp','dp_manuf','Mine_dp','Retail_dp',
            'Service_dp','Comm. Util_dp','Wholesale_dp']
IND_IC   = ['Construction','Finance','Manufacturing','Mining','retail',
            'Services','Comm. Util.','Wholesale']

# ---------------------------- 读取并合并(沿用原代码逻辑,已去冗余) -----------
def build_market_dataframe():
    fp = '/content/drive/My Drive/industry_porfolio(new).xlsx'

    df_IP = pd.read_excel(fp, sheet_name="indus_return", engine="openpyxl").iloc[1:]
    df_IP.columns = ['Year'] + IND_RET
    df_IP = df_IP.map(lambda x: str(x).replace('\xa0','') if isinstance(x,str) else x)
    df_IP = df_IP.apply(pd.to_numeric, errors='coerce')
    df_IP.iloc[:,1:] = (df_IP.iloc[:,1:]/100 + 1).round(6)

    df_ME = pd.read_excel(fp, sheet_name="ME", engine="openpyxl").iloc[1:]
    df_ME.columns = ['Year'] + [c.replace('_Ip','_ME') for c in IND_RET]

    ip = df_IP.set_index('Year');  me = df_ME.set_index('Year')
    industries = [c.replace('_Ip','') for c in ip.columns]
    R = ip.copy(); R.columns = industries
    W = me.copy(); W.columns = industries

    df_mk = pd.DataFrame(index=R.index)
    for k in industries:
        R_ex, W_ex = R.drop(columns=k), W.drop(columns=k)
        w_norm = W_ex.div(W_ex.sum(axis=1), axis=0)
        df_mk[f"R_M_excl_{k}"] = (w_norm * R_ex).sum(axis=1)
    df_mk = df_mk.reset_index()

    df1 = pd.read_excel('/content/drive/My Drive/project data/F-F_Research_Data_Factors.xlsx',
                        engine="openpyxl").iloc[1:]
    df1.columns = ['Year','Mkt','RF']
    df1['Mkt'] = df1['Mkt']/100 + 1;  df1['RF'] = df1['RF']/100 + 1

    df_If = pd.read_excel('/content/drive/My Drive/project data/CPI.xlsx',
                          sheet_name="sheet2", engine="openpyxl")
    df_If.columns = ['Year','inflation']

    # 滞后一期的收益(状态变量)
    new_df1_lag = df1[['Year','Mkt']].copy();  new_df1_lag['Year'] += 1
    df_IP_lag = df_IP.copy();                  df_IP_lag['Year'] += 1
    df_If_lag = df_If.copy();                  df_If_lag['Year'] += 1
    df_IP_lag = reduce(lambda l,r: pd.merge(l,r,on='Year',how='inner'),
                       [new_df1_lag, df_IP_lag, df_If_lag])
    df_IP_lag.columns = ['Year','Mkt_lag'] + [c+'_lag' for c in IND_RET] + ['inflation']
    lag_cols = ['Mkt_lag'] + [c+'_lag' for c in IND_RET]
    df_IP_lag[lag_cols] = df_IP_lag[lag_cols].apply(
        lambda x: x.values / df_IP_lag['inflation'].values)
    df_IP_lag = df_IP_lag.drop(columns=['inflation'])

    # 劳动收入增长率(与原代码同一文件、同一构造)
    df_IC = pd.read_excel('/content/drive/My Drive/project data/split_yearlyPerCapitaWages(new).xlsx',
                          sheet_name="sheet1", engine="openpyxl")
    df_IC.columns = ['Year'] + IND_IC + ['Government']
    ic_cols = IND_IC + ['Government']
    df_IC[ic_cols] = df_IC[ic_cols].apply(lambda x: x[1:].values / x[:-1].values)
    df_IC_lagged = df_IC.copy();  df_IC_lagged['Year'] += 1

    # dp 预测变量及其滞后
    df_dp = pd.read_excel('/content/drive/My Drive/project data/predicor_industry.xlsx',
                          sheet_name="Sheet1", usecols="A:J", engine="openpyxl")
    df_dp.columns = ["Year","dp_market"] + IND_DP
    df_dp_lag = df_dp.copy();  df_dp_lag['Year'] += 1
    df_dp_lag.columns = ["Year","dp_market_lag"] + [c+'_lag' for c in IND_DP]

    dfs = [df_IP, df_mk, df1, df_dp, df_IC_lagged, df_IP_lag, df_dp_lag, df_If]
    m = reduce(lambda l,r: pd.merge(l,r,on='Year',how='inner'), dfs)

    real_cols = IND_RET + IND_IC + ['Government','Mkt','RF'] + IND_MK
    m[real_cols] = m[real_cols].apply(lambda x: x.values / m['inflation'].values)
    m = m.drop(columns=['inflation']).iloc[:-1]
    m = m.iloc[2:]                       # 与原代码相同:去掉前两行缺失
    return m.reset_index(drop=True)

market_df = build_market_dataframe()
print("历史联合样本年数:", len(market_df))

# ---------------------------- 挑出本行业需要的列 ------------------------------
i = INDUSTRY_ID
COLS = dict(ret1=IND_RET[i], ret2=IND_MK[i], rf='RF',
            s_dp=IND_DP[i], s_lag=IND_RET[i]+'_lag', s_dplag=IND_DP[i]+'_lag',
            inc=IND_IC[i])
use_cols  = [COLS['ret1'], COLS['ret2'], COLS['rf'],           # 0-2: 三个资产的总收益
             COLS['s_dp'], COLS['s_lag'], COLS['s_dplag'],     # 3-5: 状态变量
             COLS['inc']]                                      # 6  : 劳动收入增长率
hist_arr  = market_df[use_cols].to_numpy(dtype=np.float64)

# =============================================================================
# 【修正 3】employment / retirement 不再分别独立抽样后拼接。
#   原做法在 t = K_EMPLOY 处人为切断了收益与 dp 等持续性状态变量的相依结构
#  (两段用不同随机种子、彼此独立)。这里改为:对整条 61 年路径做一次连续的
#   stationary block bootstrap,再把退休期(路径内行号 > K_EMPLOY)的收入增长
#   列置零。收益/状态的边际分布与原做法完全相同,但接缝处的序列相依性得以保留。
# =============================================================================
def stationary_block_bootstrap(data, p, n_out, rng):
    n_hist, n_col = data.shape
    out, idx = np.empty((n_out, n_col)), 0
    while idx < n_out:
        L     = rng.geometric(p)
        start = rng.integers(0, n_hist)
        block = data[start:min(start+L, n_hist)]
        block = block[:n_out-idx]
        out[idx:idx+len(block)] = block
        idx  += len(block)
    return out

rng      = np.random.default_rng(SEED)
n_rows   = (T+1) * N_PATH * N_BATCH
sim      = stationary_block_bootstrap(hist_arr, p=1/8, n_out=n_rows, rng=rng)

ttm      = np.tile(np.arange(T+1, 0, -1, dtype=np.float64), N_PATH*N_BATCH) / (T+1)
simData  = np.column_stack([sim, ttm])          # 列 7: 归一化的 time-to-maturity
# 列布局: [R_ind, R_mkt_ex, R_f, dp, R_ind_lag, dp_lag, G_income, ttm]

data3d   = simData.reshape(N_PATH*N_BATCH, T+1, 8)
data3d[:, K_EMPLOY+1:, 6] = 0.0                 # 【修正 3】退休期收入增长置零

n_tr = N_PATH*N_BATCH_TR
n_va = N_PATH*N_BATCH_VA
dataTrain = data3d[:n_tr]
dataValid = data3d[n_tr:n_tr+n_va]
dataTest  = data3d[n_tr+n_va:]
print("train/valid/test 路径数:", dataTrain.shape[0], dataValid.shape[0], dataTest.shape[0])


## 2. 策略网络 v5 与三阶段训练(算法同 v5;含【S1/S2/S3】提速)

In [8]:
# =============================================================================
# 第 2 步:策略网络、目标函数与训练(v5:纯端到端学习,无任何解析成分)
#   目标(与论文一致):  max E[ Σ_{t=0}^{T-1} β^t u(C_t) + β^T u(W_T) ],  C_T = W_T
#
# 方法定位:整条策略(消费 + 组合、工作期 + 退休期)全部由可训练参数给出,
# 训练只使用目标函数的梯度;backward induction 仅在第 3-4 节作为【外部验证
# 基准】出现,不进入方法本身。
#
# 沿用:【修正 1/2/3】【改动 A/B】(log 财富+收入输入、消费层 bias=0)、
#       【改动 D】每期消费偏置 φ_t(全生命周期 T 个)、
#       【改动 E】退休期组合向量 ψ(单个 3 维向量,20 期共享;数值由梯度学习)。
#
# v5 相对 v3/v4 的改动:
#
# 【改动 G】删除 v3 的 φ 平滑罚(其梯度 ~1.4e-4 比尾部真实梯度 ~2e-7 大约
#   600 倍,是 v3 退休段 ramp-up 消失的直接原因)。训练损失 = 真实目标,无罚项。
#
# 【改动 I】(替代 v4 的解析尾部)三阶段优化日程 —— 块坐标下降,目标函数
#   自始至终是同一个:
#     阶段 1:lr=1e-2,全部参数;
#     阶段 2:lr=1e-3,全部参数(精修);
#     阶段 3:lr=2e-2,【仅退休期参数】(φ 的退休段切片 + ψ),其余冻结。
#   理由:阶段 1/2 中尾部参数的梯度(∝ β^t X^{1-γ})在总 val_loss 里低于
#   epoch 间噪声,早停无法感知其收敛与否;阶段 3 把它们隔离出来,Adam 的
#   逐参数归一化 + 大批量梯度平均使微弱但方向稳定的尾部信号得以兑现——
#   这正是 v2 中已被实验证实有效的做法(末期比率 0.486 vs BI 0.507),
#   此处将其固化为训练流程的一部分。阶段 3 使用 2×批量以压低蒙特卡洛噪声
#   (v2 尾部锯齿的来源),checkpoint 仍以真实目标为准。
#
#   可选开关 TAIL_MONOTONE:若为 True,退休段消费比率参数化为
#   a_t = σ(φ_K + Σ softplus(δ_s))(单调递增,理论已知的形状约束;数值仍
#   由梯度学习)。默认 False,保持完全无约束的参数化。
# =============================================================================
import tensorflow as tf
import pickle, os
tf.keras.backend.set_floatx('float64')

TAIL_MONOTONE = False
N_RET = T - K_EMPLOY

class PolicyNet:
    def __init__(self, n_hidden=(10,10), n_assets=3, seed=28):
        init = lambda s: tf.keras.initializers.RandomNormal(0.0, 0.01, seed=s)
        self.h1   = tf.keras.layers.Dense(n_hidden[0], activation='tanh',
                                          kernel_initializer=init(seed))
        self.h2   = tf.keras.layers.Dense(n_hidden[1], activation='relu',
                                          kernel_initializer=init(seed+1))
        self.port = tf.keras.layers.Dense(n_assets, activation=None,
                                          kernel_initializer=init(seed+2))
        self.cons = tf.keras.layers.Dense(1, activation=None,
                                          kernel_initializer=init(seed+3),
                                          bias_initializer=tf.keras.initializers.Constant(0.0))
        self.phi  = tf.Variable(tf.zeros((T,), dtype=tf.float64), name='phi')   # 【改动 D】
        self.psi  = tf.Variable(tf.zeros((3,), dtype=tf.float64), name='psi')   # 【改动 E】
        if TAIL_MONOTONE:
            self.delta = tf.Variable(tf.fill((N_RET-1,), tf.constant(-3.0, tf.float64)),
                                     name='delta')

    def cons_logit_tail(self):
        """退休段消费 logits:默认自由;TAIL_MONOTONE 时单调递增参数化"""
        if TAIL_MONOTONE:
            base = self.phi[K_EMPLOY]
            incs = tf.math.softplus(self.delta)
            return tf.concat([[base], base + tf.cumsum(incs)], axis=0)   # (N_RET,)
        return self.phi[K_EMPLOY:]

    def heads(self, state, t, tail_logits):
        h = self.h2(self.h1(state))
        if t < K_EMPLOY:
            a = tf.sigmoid(self.cons(h) + self.phi[t])
            w = tf.nn.softmax(self.port(h), axis=1)
        else:
            a = tf.fill((tf.shape(state)[0], 1), tf.constant(0.0, tf.float64)) + \
                tf.sigmoid(tail_logits[t-K_EMPLOY])
            w = tf.fill((tf.shape(state)[0], 3), tf.constant(0.0, tf.float64)) + \
                tf.nn.softmax(self.psi)[None, :]
        return w, a

    @property
    def all_variables(self):
        v = (self.h1.trainable_weights + self.h2.trainable_weights +
             self.port.trainable_weights + self.cons.trainable_weights +
             [self.phi, self.psi])
        return v + [self.delta] if TAIL_MONOTONE else v

    @property
    def tail_variables(self):                          # 阶段 3 的参数子集
        v = [self.phi, self.psi]
        return v + [self.delta] if TAIL_MONOTONE else v

# 阶段 3 中 φ 的工作期分量不应更新:用掩码把这部分梯度置零
PHI_TAIL_MASK = tf.constant(np.concatenate([np.zeros(K_EMPLOY), np.ones(N_RET)]),
                            dtype=tf.float64)

def crra_u(c, gamma=GAMMA):
    return tf.math.log(c) if gamma == 1.0 else tf.pow(c, 1.0-gamma)/(1.0-gamma)

ADJ = tf.constant(BETA ** np.arange(T+1), dtype=tf.float64)

def simulate(data, net, collect=False):
    X = tf.fill((tf.shape(data)[0], 1), tf.constant(X0, tf.float64))
    Y = tf.fill((tf.shape(data)[0], 1), tf.constant(START_INCOME, tf.float64))
    tail_logits = net.cons_logit_tail()
    cons_seq, extra = [], {'a': [], 'w': [], 'X': [tf.identity(X)]}

    for t in range(T):
        state = tf.concat([data[:, t, 3:7],
                           tf.math.log(X / X0),
                           Y / START_INCOME], axis=1)                 # 【改动 A】
        w, a = net.heads(state, t, tail_logits)
        C  = a * X
        Rp = tf.reduce_sum(w * data[:, t+1, 0:3], axis=1, keepdims=True)
        Xn = (X - C) * Rp
        if t <= K_EMPLOY - 1:
            Y  = Y * data[:, t+1, 6:7]
            Xn = Xn + Y
        X = Xn
        cons_seq.append(C)
        if collect:
            extra['a'].append(a); extra['w'].append(w); extra['X'].append(X)

    consW = tf.concat(cons_seq + [X], axis=1)
    return (consW, extra) if collect else consW

# 【修正 1】真实目标,无 clip;【改动 G】无罚项
def objective(consW):
    tf.debugging.assert_positive(consW, message="财富/消费出现非正值——检查数据!")
    per_period = -ADJ * tf.reduce_mean(crra_u(consW), axis=0)
    return tf.reduce_sum(per_period), per_period

# =============================================================================
# 【修正 2a】+【改动 I】三阶段训练(同一目标,不同参数块/学习率/批量)
# =============================================================================
# =============================================================================
# 【提速 v5f|不改变算法】三处纯工程优化(解 T4 上 2 小时的问题):
#   S1 验证评估包进 tf.function:此前每个 epoch 在全部验证路径上以 eager
#      方式重建计算图,是运行时间的主要去向;包装后为单次图调用。
#   S2 训练批量 ×BATCH_MUL(默认 4):GPU 在 200 路径/步时严重吃不饱;
#      更大批量下梯度期望不变、噪声更小,步数按比例减少,近线性提速。
#   S3 训练中不再做任何 collect=True 的逐期监控(v5 本就没有);测试端
#      collect 只在最终验证节执行一次。
# =============================================================================
BATCH_MUL = 4

def train(net, dataTrain, dataValid,
          stages=(dict(lr=1e-2, epochs=400, patience=40, scope='all',  bmul=1),
                  dict(lr=1e-3, epochs=200, patience=25, scope='all',  bmul=1),
                  dict(lr=2e-2, epochs=300, patience=60, scope='tail', bmul=2)),
          ckpt_path='/content/drive/My Drive/pipeline_fixed/net_v5.pkl'):
    os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
    dTr, dVa = tf.constant(dataTrain), tf.constant(dataValid)
    nSeq = dataTrain.shape[0]
    best, best_w = np.inf, None

    @tf.function
    def val_fn():                                   # 【S1】验证 = 单次图调用
        return objective(simulate(dVa, net))

    for si, st in enumerate(stages):
        opt, wait = tf.keras.optimizers.Adam(st['lr'], clipnorm=1.0), 0
        tvars = net.all_variables if st['scope'] == 'all' else net.tail_variables
        bp = N_PATH * st['bmul']
        print(f"\n===== 阶段 {si+1}: lr={st['lr']}, scope={st['scope']}, batch={bp} =====")

        @tf.function
        def step(batch):
            with tf.GradientTape() as tape:
                loss, _ = objective(simulate(batch, net))
            g = tape.gradient(loss, tvars)
            if st['scope'] == 'tail':                 # 冻结 φ 的工作期分量
                g = [gi * PHI_TAIL_MASK if v is net.phi else gi
                     for gi, v in zip(g, tvars)]
            opt.apply_gradients(zip(g, tvars))
            return loss

        for ep in range(st['epochs']):
            perm = np.random.permutation(nSeq)
            for b in range(nSeq // bp):
                step(tf.gather(dTr, perm[b*bp:(b+1)*bp]))

            val_loss, val_pp = val_fn()                   # 【S1】
            val_loss = float(val_loss)
            tail_str = " ".join(f"{v:.2e}" for v in val_pp[-4:].numpy())
            print(f"epoch {ep:3d}  val={val_loss:.6f}   尾部4项(含bequest)={tail_str}")

            if val_loss < best - 1e-12:
                best, best_w, wait = val_loss, [v.numpy() for v in net.all_variables], 0
                with open(ckpt_path,'wb') as f: pickle.dump(best_w, f)
            else:
                wait += 1
                if wait >= st['patience']:
                    print("early stop");  break

        for v, w in zip(net.all_variables, best_w): v.assign(w)
    return net, best

net = PolicyNet()
_ = simulate(tf.constant(dataValid[:2]), net)
net, best_val = train(net, dataTrain, dataValid)

print("\n学到的退休期组合 softmax(ψ):", np.round(tf.nn.softmax(net.psi).numpy(), 3))
print("学到的退休期消费比率:",
      np.round(tf.sigmoid(net.cons_logit_tail()).numpy(), 3))



===== 阶段 1: lr=0.01, scope=all, batch=200 =====
epoch   0  val=0.007695   尾部4项(含bequest)=3.17e-04 9.55e-04 4.17e-03 1.58e-03
epoch   1  val=0.001424   尾部4项(含bequest)=6.75e-05 1.82e-04 7.64e-04 2.64e-04
epoch   2  val=0.000678   尾部4项(含bequest)=2.82e-05 7.29e-05 3.15e-04 1.05e-04
epoch   3  val=0.000425   尾部4项(含bequest)=1.88e-05 4.74e-05 2.01e-04 6.61e-05
epoch   4  val=0.000295   尾部4项(含bequest)=1.23e-05 3.06e-05 1.30e-04 4.25e-05
epoch   5  val=0.000220   尾部4项(含bequest)=9.63e-06 2.36e-05 9.80e-05 3.21e-05
epoch   6  val=0.000167   尾部4项(含bequest)=7.32e-06 1.77e-05 7.29e-05 2.38e-05
epoch   7  val=0.000133   尾部4项(含bequest)=5.38e-06 1.29e-05 5.32e-05 1.74e-05
epoch   8  val=0.000108   尾部4项(含bequest)=4.55e-06 1.08e-05 4.38e-05 1.42e-05
epoch   9  val=0.000090   尾部4项(含bequest)=3.43e-06 8.07e-06 3.30e-05 1.08e-05
epoch  10  val=0.000074   尾部4项(含bequest)=3.05e-06 7.10e-06 2.85e-05 9.30e-06
epoch  11  val=0.000061   尾部4项(含bequest)=2.48e-06 5.74e-06 2.27e-05 7.45e-06
epoch  12  val=0.000052   尾

KeyboardInterrupt: 

## 3. Backward induction(外部基准,逐字相同)

In [ ]:
# =============================================================================
# 第 5 步:backward induction 基准(传统方法)
#   - 退休段(无收入):问题对财富齐次,消费比率 a_t 与权重 w_t 只依赖时间,
#     可精确递推,无需财富网格;
#   - 工作段:按 x = X/Y(cash-on-hand / 当期收入)归一化后在网格上做标准
#     backward induction(Carroll / Cocco-Gomes-Maenhout 做法,亦即 Inkmann
#     2024 求解个体问题的方式)。
#   期望用与神经网络【同一个】历史联合样本 hist_arr 的经验分布计算,
#   保证两种方法面对完全相同的收益/收入分布。
# =============================================================================
from scipy.optimize import minimize

R_hist = hist_arr[:, 0:3]                       # 三资产总收益的联合历史样本
G_hist = hist_arr[:, 6]                         # 劳动收入增长率(联合行)
u = lambda c: c**(1.0-GAMMA)/(1.0-GAMMA)

simplex = ({'type':'eq','fun': lambda w: w.sum()-1.0},)
bnds3   = [(0.0,1.0)]*3

# ---------- 退休段:精确比率递推 ----------------------------------------------
def solve_retirement(n_ret=T-K_EMPLOY):
    Phi, a_path, w_path, w0 = 1.0, np.zeros(n_ret), np.zeros((n_ret,3)), np.array([.3,.3,.4])
    for t in range(n_ret-1, -1, -1):
        f  = lambda w: -np.mean(u(R_hist @ w))              # max E[u(R^p)]
        rs = minimize(f, w0, bounds=bnds3, constraints=simplex, method='SLSQP')
        ERp1g = -rs.fun * (1.0-GAMMA)                       # E[(R^p)^{1-γ}]
        k  = (BETA*Phi*ERp1g)**(1.0/GAMMA)
        a  = 1.0/(1.0+k)
        Phi = a**(1-GAMMA) + BETA*Phi*ERp1g*(1-a)**(1-GAMMA)
        a_path[t], w_path[t], w0 = a, rs.x, rs.x
    return a_path, w_path, Phi                              # Phi = 退休时点的值函数系数

a_ret, w_ret, Phi_K = solve_retirement()
print("退休段最优消费比率(t=K..T-1):\n", np.round(a_ret,3))

# ---------- 工作段:x = X/Y 网格 -----------------------------------------------
x_grid = np.concatenate([np.linspace(0.2, 5, 60), np.linspace(5.3, 60, 60)])
V_next = Phi_K * u(x_grid)                                  # V_K(x) = Phi_K·u(x)
polA   = np.zeros((K_EMPLOY, len(x_grid)))                  # a(t, x)
polW   = np.zeros((K_EMPLOY, len(x_grid), 3))               # w(t, x)

for t in range(K_EMPLOY-1, -1, -1):
    V_now = np.zeros_like(x_grid)
    p0 = np.array([0.3, .3, .3, .4])
    for j, x in enumerate(x_grid):
        def neg_obj(p):
            a, w = p[0], p[1:]
            c, s = a*x, (1-a)*x
            xn   = s*(R_hist @ w)/G_hist + 1.0              # 归一化预算约束
            cont = np.mean(G_hist**(1.0-GAMMA) * np.interp(xn, x_grid, V_next))
            return -(u(c) + BETA*cont)
        rs = minimize(neg_obj, p0, method='SLSQP',
                      bounds=[(1e-3,0.999)]+bnds3,
                      constraints=({'type':'eq','fun':lambda p: p[1:].sum()-1.0},))
        polA[t,j], polW[t,j], V_now[j] = rs.x[0], rs.x[1:], -rs.fun
        p0 = rs.x
    V_next = V_now
    if t % 5 == 0: print(f"backward induction: 工作段 t={t} 完成")

# ---------- 在同一组测试路径上模拟 BI 策略(与 NN 完全可比) --------------------
def simulate_BI(data3d_test):
    n  = data3d_test.shape[0]
    X  = np.full(n, X0);  Y = np.full(n, START_INCOME)
    Cs, Xs, Ws, As = [], [X.copy()], [], []
    for t in range(T):
        if t < K_EMPLOY:
            x   = X/Y
            a   = np.interp(x, x_grid, polA[t])
            w   = np.stack([np.interp(x, x_grid, polW[t,:,k]) for k in range(3)],1)
            w  /= w.sum(1, keepdims=True)
        else:
            a   = np.full(n, a_ret[t-K_EMPLOY])
            w   = np.tile(w_ret[t-K_EMPLOY], (n,1))
        C   = a*X
        Rp  = (w * data3d_test[:, t+1, 0:3]).sum(1)
        X   = (X - C)*Rp
        if t <= K_EMPLOY-1:
            Y = Y * data3d_test[:, t+1, 6];  X = X + Y
        Cs.append(C); Xs.append(X.copy()); Ws.append(w.mean(0)); As.append(a.mean())
    return (np.array(Cs).T, np.array(Xs).T, np.array(Ws), np.array(As))

C_bi, X_bi, W_bi, A_bi = simulate_BI(dataTest)
print("BI 平均 bequest:", X_bi[:,-1].mean().round(2))


## 4. 验证与图表(V1–V5)

In [ ]:
# =============================================================================
# 第 6 步:验证与图表 —— 神经网络(v3) vs backward induction
#   V1 末期消费比率应复现 BI 的 ramp-up(最后一年 ≈ 0.5);
#   V2 平均 bequest 收敛到 BI 量级;
#   V3 CE 差距(BI 为上界)——重点观察工作期误差是否较 v2 的 ~8% 收窄;
#   V4 组合权重:退休段应为常数向量(【改动 E】),与 BI 的 0.47/0.53 对照;
#      注意:借款约束期(早年 a≈0.99)储蓄≈0,目标对权重几乎平坦,
#      该区间两法的权重差异【没有经济含义】,图中以灰色底纹标出;
#   V5 可行性:财富恒正、权重在单纯形内。
# =============================================================================
import matplotlib.pyplot as plt

dTe = tf.constant(dataTest)
cw_nn, ex = simulate(dTe, net, collect=True)
cw_nn  = cw_nn.numpy()
A_nn   = np.array([a.numpy().mean()  for a in ex['a']])
W_nn   = np.array([w.numpy().mean(0) if w.shape[0] > 1 else w.numpy()[0] for w in ex['w']])
X_nn   = np.concatenate([x.numpy() for x in ex['X']], axis=1)

def ce(consW):
    U = float(np.sum(ADJ.numpy() * np.mean(consW**(1-GAMMA)/(1-GAMMA), axis=0)))
    return ((1-GAMMA)*U/ADJ.numpy().sum())**(1/(1-GAMMA))

consW_bi = np.column_stack([C_bi, X_bi[:,-1]])
ages     = np.arange(T)
CONSTR_END = int(np.argmax(A_bi < 0.95))        # 借款约束期的近似右端点

fig, ax = plt.subplots(2, 2, figsize=(13, 9))

ax[0,0].plot(ages, A_bi, 'k-', lw=2, label='backward induction')
ax[0,0].plot(ages, A_nn, 'C0-', label='NN')
ax[0,0].axvline(K_EMPLOY, color='grey', ls=':')
ax[0,0].set_title('Consumption / cash-on-hand ratio');  ax[0,0].legend()

ax[0,1].plot(range(T+1), X_bi.mean(0), 'k-', lw=2, label='BI')
ax[0,1].plot(range(T+1), X_nn.mean(0), 'C0-', label='NN')
ax[0,1].axvline(K_EMPLOY, color='grey', ls=':')
ax[0,1].set_title('Mean cash-on-hand path');  ax[0,1].legend()

ax[1,0].plot(ages, C_bi.mean(0),      'k-', lw=2, label='BI')
ax[1,0].plot(ages, cw_nn[:,:T].mean(0),'C0-', label='NN')
ax[1,0].set_title('Mean consumption path');  ax[1,0].legend()

lbl = ['own industry', 'market ex-industry', 'risk-free']
for k in range(3):
    ax[1,1].plot(ages, W_bi[:,k], color=f'C{k}', lw=2, label=f'BI {lbl[k]}')
    ax[1,1].plot(ages, W_nn[:,k], color=f'C{k}', ls='--', label=f'NN {lbl[k]}')
ax[1,1].axvspan(0, CONSTR_END, color='grey', alpha=0.15,
                label='borrowing-constrained (weights ~irrelevant)')
ax[1,1].set_title('Mean portfolio weights (solid BI, dashed NN)')
ax[1,1].legend(fontsize=7)
plt.tight_layout();  plt.savefig('nn_vs_backward_induction_v5.png', dpi=200);  plt.show()

print("="*72)
print("V1 末期(T-1)消费比率:   BI = %.3f | NN v5 = %.3f" % (A_bi[-1], A_nn[-1]))
print("V2 平均 bequest W_T:      BI = %.1f | NN v5 = %.1f"
      % (X_bi[:,-1].mean(), X_nn[:,-1].mean()))
ce_bi, ce_nn = ce(consW_bi), ce(cw_nn)
print("V3 确定性等价消费 CE:     BI = %.3f | NN v5 = %.3f  (福利损失 %.2f%%)"
      % (ce_bi, ce_nn, 100*(1-ce_nn/ce_bi)))
print("V4 退休期权重 (NN,常数): ", np.round(W_nn[-1], 3),
      " | BI:", np.round(W_bi[-1], 3))
print("V5 财富为正:", bool((X_nn > 0).all()), " | 权重和为 1:",
      bool(np.allclose(W_nn.sum(1), 1.0, atol=1e-6)))
print("="*72)
print("提示:图 D 灰色区为借款约束期(储蓄≈0),两法的权重差异在该区间无经济含义。")
